# GPT-2 classifier with a variable-position readout

This notebook fine-tunes [`openai-community/gpt2`](https://huggingface.co/openai-community/gpt2) as a binary classifier on the [`rasbt/human-vs-ai-50k`](https://huggingface.co/datasets/rasbt/human-vs-ai-50k) dataset. GPT-2's pretrained end-of-text token is appended immediately after each text and used as the classification readout. Its absolute position varies with the input length.

A dedicated padding token is added so that padding and the end-of-text readout remain distinct. Batches are padded dynamically to their longest sequence. This optimized version requires Linux, CUDA, and FlashAttention 2. From the project directory, run `uv sync --group dev`, install its build requirements with `uv pip install packaging psutil ninja`, and then run `MAX_JOBS=4 uv pip install flash-attn --no-build-isolation`. Open the notebook with `uv run jupyter lab scripts/10_gpt2/gpt2-variable-position.ipynb`.

In [ ]:
import json
import platform
from importlib.metadata import version
from importlib.util import find_spec
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    confusion_matrix,
    log_loss,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

In [ ]:
print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Device: {device}")
print(f"FlashAttention installed: {find_spec('flash_attn') is not None}")

## Load the dataset

In [ ]:
dataset = load_dataset("rasbt/human-vs-ai-50k")

train_dataset = dataset["train"]
validation_dataset = dataset["validation"]
test_dataset = dataset["test"]

dataset

In [ ]:
split_summary = pd.DataFrame(
    [
        {
            "split": split_name,
            "human": split_dataset["label"].count(0),
            "ai": split_dataset["label"].count(1),
            "total": len(split_dataset),
        }
        for split_name, split_dataset in dataset.items()
    ]
).set_index("split")

split_summary

## Variable-position tokenization

GPT-2 supports at most 1,024 positions. Each text is therefore truncated to 1,023 tokens, leaving one position for `<|endoftext|>`. The readout token is appended directly after the text. Dynamic right-padding is applied only when a batch is assembled.

In [ ]:
MODEL_NAME = "openai-community/gpt2"
PAD_TOKEN = "<|pad|>"
READOUT_POSITION = "variable"
CONTEXT_LENGTH = 1024
MAX_TEXT_LENGTH = CONTEXT_LENGTH - 1
RANDOM_STATE = 17

set_seed(RANDOM_STATE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.add_special_tokens({"pad_token": PAD_TOKEN})
tokenizer.padding_side = "right"


def encode_variable_texts(texts, active_tokenizer):
    encoded = active_tokenizer(
        texts,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_TEXT_LENGTH,
    )
    eos_token_id = active_tokenizer.eos_token_id
    return {
        "input_ids": [
            input_ids + [eos_token_id]
            for input_ids in encoded["input_ids"]
        ],
        "attention_mask": [
            attention_mask + [1]
            for attention_mask in encoded["attention_mask"]
        ],
    }


def tokenize_batch(batch):
    return encode_variable_texts(batch["text"], tokenizer)


columns_to_remove = [
    column
    for column in train_dataset.column_names
    if column != "label"
]

tokenized_dataset = dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=columns_to_remove,
    desc="Tokenizing",
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

tokenized_dataset

In [ ]:
example = tokenized_dataset["train"][0]
assert example["input_ids"][-1] == tokenizer.eos_token_id
assert example["attention_mask"][-1] == 1
print(f"Example length including readout: {len(example['input_ids'])}")

## Fine-tuning

The entire GPT-2 model and classification head are trained on the training split. Validation accuracy selects the best epoch, and early stopping ends training when validation accuracy stops improving. The test split remains untouched until the final evaluation.

FlashAttention 2 supplies the optimized attention kernels. Length grouping reduces padding within batches, while PyTorch compilation, fused AdamW, and persistent data-loader workers reduce additional overhead.

In [ ]:
id2label = {0: "HUMAN", 1: "AI"}
label2id = {"HUMAN": 0, "AI": 1}

if not torch.cuda.is_available():
    raise RuntimeError(
        "This optimized notebook requires a CUDA GPU."
    )
if find_spec("flash_attn") is None:
    raise ImportError(
        "FlashAttention 2 is required. Install it with "
        "`uv pip install packaging psutil ninja`, followed by "
        "`MAX_JOBS=4 uv pip install flash-attn --no-build-isolation`."
    )

ATTN_IMPLEMENTATION = "flash_attention_2"
use_bf16 = torch.cuda.is_bf16_supported()
model_dtype = torch.bfloat16 if use_bf16 else torch.float16

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    attn_implementation=ATTN_IMPLEMENTATION,
    dtype=model_dtype,
)
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)
print(f"Trainable parameters: {trainable_parameters:,}")
print(f"Attention implementation: {model.config._attn_implementation}")
print(f"Model dtype: {model_dtype}")


def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, predictions)}


training_args = TrainingArguments(
    output_dir="checkpoints-gpt2-variable-position",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_steps=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=2,
    bf16=use_bf16,
    fp16=not use_bf16,
    optim="adamw_torch_fused",
    train_sampling_strategy="group_by_length",
    torch_compile=True,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    dataloader_persistent_workers=True,
    dataloader_prefetch_factor=2,
    report_to="none",
    seed=RANDOM_STATE,
    data_seed=RANDOM_STATE,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

In [ ]:
train_result = trainer.train()

print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Best validation accuracy: {trainer.state.best_metric:.2%}")
print(
    "Training time: "
    f"{train_result.metrics['train_runtime'] / 60:.1f} minutes"
)

## Temperature scaling

Temperature scaling learns one positive scalar from the validation logits. It changes probability confidence without changing which class has the larger logit.

In [ ]:
trainer.args.train_sampling_strategy = "sequential"
validation_output = trainer.predict(
    tokenized_dataset["validation"]
)
validation_logits = np.asarray(validation_output.predictions)
y_validation = np.asarray(
    validation_output.label_ids, dtype=np.int64
)


def fit_temperature(logits, labels):
    logits_tensor = torch.tensor(logits, dtype=torch.float64)
    labels_tensor = torch.tensor(labels, dtype=torch.long)
    log_temperature = torch.nn.Parameter(
        torch.zeros((), dtype=torch.float64)
    )
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.LBFGS(
        [log_temperature],
        lr=0.1,
        max_iter=50,
        line_search_fn="strong_wolfe",
    )

    def closure():
        optimizer.zero_grad()
        temperature = log_temperature.exp()
        loss = criterion(logits_tensor / temperature, labels_tensor)
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(log_temperature.exp().detach())


def probabilities_from_logits(logits, temperature=1.0):
    scaled_logits = np.asarray(logits, dtype=np.float64) / temperature
    scaled_logits -= scaled_logits.max(axis=1, keepdims=True)
    exponentiated = np.exp(scaled_logits)
    return exponentiated / exponentiated.sum(axis=1, keepdims=True)


TEMPERATURE = fit_temperature(validation_logits, y_validation)
validation_accuracy = accuracy_score(
    y_validation, np.argmax(validation_logits, axis=1)
)
print(f"Validation accuracy: {validation_accuracy:.2%}")
print(f"Learned temperature: {TEMPERATURE:.4f}")

In [ ]:
uncalibrated_validation_probability = probabilities_from_logits(
    validation_logits
)[:, 1]
calibrated_validation_probability = probabilities_from_logits(
    validation_logits, TEMPERATURE
)[:, 1]

calibration_comparison = pd.DataFrame(
    [
        {
            "model": "Uncalibrated",
            "Brier score": brier_score_loss(
                y_validation, uncalibrated_validation_probability
            ),
            "Log loss": log_loss(
                y_validation, uncalibrated_validation_probability
            ),
        },
        {
            "model": "Temperature-scaled",
            "Brier score": brier_score_loss(
                y_validation, calibrated_validation_probability
            ),
            "Log loss": log_loss(
                y_validation, calibrated_validation_probability
            ),
        },
    ]
).set_index("model")

calibration_comparison.style.format("{:.4f}")

## Final evaluation

The positive class is AI-generated text. The held-out test split is evaluated only after checkpoint selection and temperature scaling.

In [ ]:
DECISION_THRESHOLD = 0.5

split_logits = {"validation": validation_logits}
split_labels = {"validation": y_validation}
for split_name in ("train", "test"):
    prediction_output = trainer.predict(
        tokenized_dataset[split_name]
    )
    split_logits[split_name] = np.asarray(
        prediction_output.predictions
    )
    split_labels[split_name] = np.asarray(
        prediction_output.label_ids, dtype=np.int64
    )

for split_name in ("train", "validation", "test"):
    np.testing.assert_array_equal(
        split_labels[split_name],
        np.asarray(dataset[split_name]["label"], dtype=np.int64),
    )


def evaluate_split(split_name):
    y_true = split_labels[split_name]
    ai_probability = probabilities_from_logits(
        split_logits[split_name], TEMPERATURE
    )[:, 1]
    y_pred = (
        ai_probability >= DECISION_THRESHOLD
    ).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()
    return {
        "split": split_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "brier_score": brier_score_loss(y_true, ai_probability),
        "log_loss": log_loss(y_true, ai_probability),
        "true_negative": tn,
        "false_positive": fp,
        "false_negative": fn,
        "true_positive": tp,
    }


evaluation = pd.DataFrame(
    [evaluate_split(name) for name in ("train", "validation", "test")]
).set_index("split")

evaluation.style.format(
    {
        "accuracy": "{:.2%}",
        "brier_score": "{:.4f}",
        "log_loss": "{:.4f}",
    }
)

## Test-set confusion matrix

In [ ]:
y_test = split_labels["test"]
y_test_ai_probability = probabilities_from_logits(
    split_logits["test"], TEMPERATURE
)[:, 1]
y_test_pred = (
    y_test_ai_probability >= DECISION_THRESHOLD
).astype(np.int64)

test_matrix = confusion_matrix(y_test, y_test_pred, labels=[1, 0])
cell_labels = np.asarray(
    [
        ["True positive (TP)", "False negative (FN)"],
        ["False positive (FP)", "True negative (TN)"],
    ]
)

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(test_matrix, cmap="Blues", vmin=0, vmax=test_matrix.max())

for row in range(2):
    for column in range(2):
        value = test_matrix[row, column]
        text_color = (
            "white" if value > test_matrix.max() / 2 else "#222222"
        )
        ax.text(
            column, row - 0.08, f"{value:,}",
            ha="center", va="center",
            color=text_color, fontsize=24,
        )
        ax.text(
            column, row + 0.14, cell_labels[row, column],
            ha="center", va="center",
            color=text_color, fontsize=10,
            fontweight=(
                "bold" if value > test_matrix.max() / 2 else "normal"
            ),
        )

ax.set_xticks([0, 1], labels=["AI", "Human"])
ax.set_yticks([0, 1], labels=["AI", "Human"])
ax.set_xlabel("Model prediction", fontsize=13, fontweight="bold", labelpad=15)
ax.set_ylabel("Actual label", fontsize=13, fontweight="bold", labelpad=15)
ax.axhline(0.5, color="#d0d0d0", linewidth=1)
ax.axvline(0.5, color="#d0d0d0", linewidth=1)
ax.tick_params(axis="both", labelsize=11, pad=7)
for spine in ax.spines.values():
    spine.set_color("#555555")
    spine.set_linewidth(1)

fig.tight_layout()
plt.savefig("gpt2-variable-position-confmat.svg")
plt.show()

## Inspect confident test-set errors

This table helps identify source-specific failures and possible dataset artifacts.

In [ ]:
test_errors = pd.DataFrame(
    {
        "id": test_dataset["id"],
        "actual": np.where(y_test == 1, "AI", "Human"),
        "prediction": np.where(y_test_pred == 1, "AI", "Human"),
        "ai_probability": y_test_ai_probability,
        "source_collection": test_dataset["source_collection"],
        "generator_model": test_dataset["generator_model"],
        "text": [text[:300] for text in test_dataset["text"]],
    }
)
test_errors = test_errors[
    test_errors["actual"] != test_errors["prediction"]
].copy()
test_errors["prediction_confidence"] = np.where(
    test_errors["prediction"] == "AI",
    test_errors["ai_probability"],
    1.0 - test_errors["ai_probability"],
)

test_errors.sort_values(
    "prediction_confidence", ascending=False
).head(20).style.format(
    {
        "ai_probability": "{:.2%}",
        "prediction_confidence": "{:.2%}",
    }
)

## Export the calibrated model

The exported directory contains the best checkpoint, tokenizer, readout configuration, learned temperature, and decision threshold.

In [ ]:
ARTIFACT_DIR = (
    Path("artifacts") / "gpt2-variable-position-ai-detector"
)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(ARTIFACT_DIR)
tokenizer.save_pretrained(ARTIFACT_DIR)

metadata = {
    "dataset": "rasbt/human-vs-ai-50k",
    "base_model": MODEL_NAME,
    "label_mapping": {"human": 0, "ai": 1},
    "readout_position": READOUT_POSITION,
    "attention_implementation": ATTN_IMPLEMENTATION,
    "model_dtype": str(model_dtype),
    "readout_token": tokenizer.eos_token,
    "pad_token": tokenizer.pad_token,
    "context_length": CONTEXT_LENGTH,
    "max_text_length": MAX_TEXT_LENGTH,
    "decision_threshold": DECISION_THRESHOLD,
    "calibration": "temperature scaling",
    "temperature": TEMPERATURE,
    "best_validation_accuracy": float(trainer.state.best_metric),
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "trainable_parameters": trainable_parameters,
    "per_device_train_batch_size": training_args.per_device_train_batch_size,
    "gradient_accumulation_steps": training_args.gradient_accumulation_steps,
    "training_runtime_seconds": train_result.metrics["train_runtime"],
    "random_state": RANDOM_STATE,
    "python_version": platform.python_version(),
    "package_versions": {
        package: version(package)
        for package in [
            "datasets", "numpy", "scikit-learn",
            "torch", "transformers",
        ]
    },
}

METADATA_PATH = ARTIFACT_DIR / "detector-config.json"
METADATA_PATH.write_text(
    json.dumps(metadata, indent=2) + "\n", encoding="utf-8"
)

print(f"Saved model and tokenizer to {ARTIFACT_DIR}")
print(f"Saved detector configuration to {METADATA_PATH}")

## Load the model in a script

The helper recreates the variable-position end-of-text readout before returning calibrated AI probabilities.

In [ ]:
loaded_tokenizer = AutoTokenizer.from_pretrained(ARTIFACT_DIR)
loaded_model = AutoModelForSequenceClassification.from_pretrained(
    ARTIFACT_DIR,
    attn_implementation=ATTN_IMPLEMENTATION,
    dtype=model_dtype,
)
loaded_config = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
loaded_model.to(device)
loaded_model.eval()

loaded_collator = DataCollatorWithPadding(
    tokenizer=loaded_tokenizer,
    pad_to_multiple_of=8 if device.type == "cuda" else None,
)


def prepare_variable_inputs(texts):
    encoded = loaded_tokenizer(
        texts,
        add_special_tokens=False,
        truncation=True,
        max_length=loaded_config["max_text_length"],
    )
    features = []
    for input_ids, attention_mask in zip(
        encoded["input_ids"], encoded["attention_mask"]
    ):
        features.append(
            {
                "input_ids": input_ids + [loaded_tokenizer.eos_token_id],
                "attention_mask": attention_mask + [1],
            }
        )
    return loaded_collator(features)


def predict_ai_probability(texts):
    inputs = {
        key: value.to(device)
        for key, value in prepare_variable_inputs(texts).items()
    }
    with torch.inference_mode():
        logits = loaded_model(**inputs).logits
        calibrated_logits = logits / loaded_config["temperature"]
        return (
            calibrated_logits.softmax(dim=1)[:, 1]
            .float()
            .cpu()
            .numpy()
        )


texts = ["Replace this string with the text you want to classify."]
ai_probabilities = predict_ai_probability(texts)
for text, ai_probability in zip(texts, ai_probabilities):
    prediction = int(
        ai_probability >= loaded_config["decision_threshold"]
    )
    label = "AI" if prediction == 1 else "Human"
    print(f"{label}: {ai_probability:.1%} AI probability | {text}")